# Cattle Re-ID — Etapa 3: evaluación final (dos protocolos)

Re-evaluación determinista de todos los checkpoints (no reentrena). Reusa `scripts/reid_eval.py`.
Se reportan **dos protocolos** con datos reales (nada fabricado):

- **Protocolo A — crudo + mcs=4:** dataset sin dedup (4923 imgs). En crudo el mínimo real es 4
  fotos/vaca, así que mcs=4 no disuelve ninguna. Es un **límite superior OPTIMISTA**: los
  near-duplicates de ráfaga facilitan el clustering (sin control anti-fuga).
- **Protocolo B — dedup + mcs=2:** pHash dedup (1554 imgs) + mcs=2. El número **limpio y
  desplegable** (anti-fuga). **Es el headline.**

El delta A−B cuantifica cuánto inflaban los duplicados. eps **label-free** (silhouette, grid ≤ 0.05).
Métricas completas de `reid_eval.py`. Todo a Drive (`outputs/results/eval_final/`).

## 1. Montar Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/cattle_reid')
assert DRIVE_ROOT.is_dir(), f'No existe {DRIVE_ROOT}.'
print('contenido:', [p.name for p in DRIVE_ROOT.iterdir()])

Mounted at /content/drive
contenido: ['BeefCattle_Muzzle_Individualized.zip', 'outputs', 'dataset_caras_bovinos.zip', 'gradcam_test', 'CMPD300_Baseline.zip', 'Untitled0.ipynb', 'model_cache', 'Untitled']


## 2. Cachear modelos en Drive (antes de importar torch)

In [2]:
import os
CACHE = DRIVE_ROOT / 'model_cache'
(CACHE/'torch').mkdir(parents=True, exist_ok=True)
(CACHE/'hf').mkdir(parents=True, exist_ok=True)
os.environ['TORCH_HOME'] = str(CACHE/'torch')
os.environ['HF_HOME']    = str(CACHE/'hf')
print('cache ->', CACHE)

cache -> /content/drive/MyDrive/cattle_reid/model_cache


## 3. GPU

In [3]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), 'Sin GPU: Entorno -> Cambiar tipo -> GPU.'
print('GPU:', torch.cuda.get_device_name(0))

GPU 0: NVIDIA L4 (UUID: GPU-2596ba86-9a2f-ed63-72aa-22eeab195ba9)
GPU: NVIDIA L4


## 4. Código + dependencias

In [4]:
import shutil
os.chdir('/content')
REPO_URL = 'https://github.com/benjaminvitale/tp-final-vision2-Pujol-Vitale.git'
REPO_DIR = '/content/tp-final-vision2-Pujol-Vitale'
if os.path.exists(REPO_DIR): shutil.rmtree(REPO_DIR)
!git clone -b main {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1
!pip -q install 'transformers==4.40.2' seaborn
import sys; sys.path.insert(0, 'scripts')   # para importar reid_eval

Cloning into '/content/tp-final-vision2-Pujol-Vitale'...
remote: Enumerating objects: 403, done.
remote: Counting objects: 100% (98/98), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 403 (delta 50), reused 48 (delta 24), pack-reused 305 (from 1)
Receiving objects: 100% (403/403), 365.72 KiB | 1.77 MiB/s, done.
Resolving deltas: 100% (225/225), done.
/content/tp-final-vision2-Pujol-Vitale
0725118 (HEAD -> main, origin/main, origin/HEAD) Etapa 3 eval: dos protocolos (A crudo+mcs4 optimista / B dedup+mcs2 limpio)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 140.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the 

## 5. Persistir outputs en Drive (symlinks)

In [5]:
for sub in ('checkpoints', 'logs', 'results'):
    ds = DRIVE_ROOT / 'outputs' / sub; ds.mkdir(parents=True, exist_ok=True)
    ls = Path(REPO_DIR) / 'outputs' / sub
    if ls.exists() and not ls.is_symlink(): shutil.rmtree(ls)
    if not ls.exists(): os.symlink(ds, ls, target_is_directory=True)
EVALDIR = Path('outputs/results/eval_final'); EVALDIR.mkdir(parents=True, exist_ok=True)
print('salidas ->', DRIVE_ROOT / 'outputs' / 'results' / 'eval_final')

salidas -> /content/drive/MyDrive/cattle_reid/outputs/results/eval_final


## 6. Target (Zenodo) — dos sets: crudo y dedup + diagnóstico de fotos/vaca

In [7]:
import numpy as np, csv
MUZZLE_ZIP = DRIVE_ROOT / 'BeefCattle_Muzzle_Individualized.zip'
MUZZLE_LOCAL = Path('/content/muzzle')
if not MUZZLE_LOCAL.exists():
    MUZZLE_LOCAL.mkdir(parents=True)
    !unzip -q "{MUZZLE_ZIP}" -d "{MUZZLE_LOCAL}"
def find_root(base):
    for p in [base, *[d for d in base.rglob('*') if d.is_dir()]]:
        subs = [d for d in p.iterdir() if d.is_dir()]
        if len(subs) >= 100 and any(d.name.lower().startswith('cattle') for d in subs): return p
    raise RuntimeError('No encuentro cattle_*')
TARGET_DIR = find_root(MUZZLE_LOCAL)

from src.reid.reid_dataset import entries_from_folders
from src.reid.phash import dedup_entries

entries_raw, _ = entries_from_folders(TARGET_DIR)                 # A: crudo, con duplicados
entries_dedup, dinfo = dedup_entries(list(entries_raw), TARGET_DIR, threshold=6)  # B: sin duplicados

def diag(entries, tag):
    lab = np.array([e['label'] for e in entries]); nt = len(np.unique(lab))
    _, cts = np.unique(lab, return_counts=True)
    print(f'[{tag}] {len(entries)} imgs | {nt} ids | fotos/vaca min={cts.min()} mediana={int(np.median(cts))} max={cts.max()}')
    for k in (2, 4):
        print(f'    vacas con < {k} fotos: {(cts < k).sum()}/{nt} ({100*(cts<k).sum()/nt:.1f}%)')
    return lab, nt
lab_raw, nt_raw = diag(entries_raw, 'A crudo')
lab_dedup, nt_dedup = diag(entries_dedup, 'B dedup'); print('dedup:', dinfo)

# indices reproducibles
for entries, nm in [(entries_raw, 'index_raw.csv'), (entries_dedup, 'index_dedup.csv')]:
    with open(EVALDIR / nm, 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['path', 'label'])
        for e in entries: w.writerow([str(e['path']), e['label']])
print('indices guardados en', EVALDIR)

[A crudo] 4923 imgs | 268 ids | fotos/vaca min=4 mediana=16 max=70
    vacas con < 2 fotos: 0/268 (0.0%)
    vacas con < 4 fotos: 0/268 (0.0%)
[B dedup] 1554 imgs | 268 ids | fotos/vaca min=1 mediana=5 max=27
    vacas con < 2 fotos: 8/268 (3.0%)
    vacas con < 4 fotos: 78/268 (29.1%)
dedup: {'n_input': 4923, 'n_kept': 1554, 'n_dropped': 3369, 'threshold': 6}
indices guardados en outputs/results/eval_final


## 7. Config: los dos protocolos

In [8]:
CONFIGS = [
    dict(name='A_crudo_mcs4',  entries=entries_raw,   lab=lab_raw,   n_true=nt_raw,   mcs=4,
         desc='crudo (con duplicados), mcs=4 - OPTIMISTA / sin anti-fuga'),
    dict(name='B_dedup_mcs2',  entries=entries_dedup, lab=lab_dedup, n_true=nt_dedup, mcs=2,
         desc='dedup (sin duplicados), mcs=2 - LIMPIO / desplegable (headline)'),
]
EPS_GRID = list(np.round(np.arange(0.0, 0.051, 0.005), 3))   # grid <= 0.05, sin floor
SEED = 0
for c in CONFIGS: print(c['name'], '->', c['desc'])

A_crudo_mcs4 -> crudo (con duplicados), mcs=4 - OPTIMISTA / sin anti-fuga
B_dedup_mcs2 -> dedup (sin duplicados), mcs=2 - LIMPIO / desplegable (headline)


## 8. Sanity gate (Protocolo B, mcs=2) — debe dar ~0.461 (imagenet) y ~0.716 (dinov2L)

In [9]:
from sklearn.cluster import HDBSCAN
from src.reid.encoders import resnet50_checkpoint, dinov2_checkpoint, build_encoder
from src.reid.cluster import clustering_metrics

def embed(enc, entries):
    return enc.embed(entries, data_dir=TARGET_DIR, batch_size=64, num_workers=2)

for name, enc, exp in [('imagenet', build_encoder('imagenet'), 0.461),
                       ('dinov2L', dinov2_checkpoint('outputs/checkpoints/cmpd300_supcon_dinov2L_strong.pt'), 0.716)]:
    e, l = embed(enc, entries_dedup)
    ari = clustering_metrics(l, HDBSCAN(min_cluster_size=2, metric='cosine').fit_predict(e))['ARI']
    print(f'  {name:10} ARI@eps0(dedup,mcs=2)={ari:.3f} (esp ~{exp}) {"OK" if abs(ari-exp)<0.02 else "REVISAR"}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[16:33:24] INFO encoder='imagenet_resnet50' embedded 1554 imgs → dim 2048
INFO:reid.encoders:encoder='imagenet_resnet50' embedded 1554 imgs → dim 2048


  imagenet   ARI@eps0(dedup,mcs=2)=0.461 (esp ~0.461) OK


[16:34:06] INFO encoder='cmpd300_supcon_dinov2L_strong.pt' embedded 1554 imgs → dim 1024
INFO:reid.encoders:encoder='cmpd300_supcon_dinov2L_strong.pt' embedded 1554 imgs → dim 1024


  dinov2L    ARI@eps0(dedup,mcs=2)=0.716 (esp ~0.716) OK


## 9. Evaluar todos los modelos en los DOS protocolos (batería completa)

In [10]:
import json
from scripts.reid_eval import full_metrics, eps_sweep
from src.reid.eval_reid import rank_metrics
from src.reid.cluster import kmeans_reference
import random

def single_shot_idx(labels, seed=0):
    rng = random.Random(seed); by = {}
    for i, l in enumerate(labels.tolist()): by.setdefault(l, []).append(i)
    gal, prb = [], []
    for _l, idxs in sorted(by.items()):
        if len(idxs) < 2: continue
        idxs = idxs[:]; rng.shuffle(idxs); gal.append(idxs[0]); prb += idxs[1:]
    return gal, prb

def loader(ckpt, base):
    if ckpt is None:
        return build_encoder('dinov2' if base == 'dinov2' else 'imagenet')
    return dinov2_checkpoint(ckpt) if base == 'dinov2' else resnet50_checkpoint(ckpt)

# (display, ckpt|None, base, aug, grupo)
ALL_MODELS = [
    ('ImageNet (frozen)',            None,                                                    'resnet', '-',      'baseline'),
    ('DINOv2-base (frozen)',         None,                                                    'dinov2', '-',      'baseline'),
    ('CE',                           'outputs/checkpoints/cmpd300_ce.pt',                     'resnet', 'strong', 'loss'),
    ('ArcFace',                      'outputs/checkpoints/cmpd300_arcface.pt',                'resnet', 'strong', 'loss'),
    ('Triplet',                      'outputs/checkpoints/cmpd300_triplet.pt',                'resnet', 'strong', 'loss'),
    ('SupCon (ResNet, strong)',      'outputs/checkpoints/cmpd300_supcon.pt',                 'resnet', 'strong', 'both'),
    ('SupCon (ResNet, heavy)',       'outputs/checkpoints/cmpd300_supcon_heavyaug.pt',        'resnet', 'heavy',  'model'),
    ('SupCon (DINOv2-base, strong)', 'outputs/checkpoints/cmpd300_supcon_dinov2_strong.pt',   'dinov2', 'strong', 'model'),
    ('SupCon (DINOv2-base, heavy)',  'outputs/checkpoints/cmpd300_supcon_dinov2_heavyaug.pt', 'dinov2', 'heavy',  'model'),
    ('SupCon (DINOv2-large, strong)','outputs/checkpoints/cmpd300_supcon_dinov2L_strong.pt',  'dinov2', 'strong', 'model'),
]

RESULTS = {c['name']: {} for c in CONFIGS}
META = {disp: dict(base=base, aug=aug, grupo=grupo) for disp, _, base, aug, grupo in ALL_MODELS}

for cfg in CONFIGS:
    print(f"\n===== {cfg['name']} ({cfg['desc']}) =====")
    ents, lb, mcs = cfg['entries'], cfg['lab'], cfg['mcs']
    gal, prb = single_shot_idx(lb, SEED)
    for disp, ckpt, base, aug, grupo in ALL_MODELS:
        if ckpt is not None and not Path(ckpt).is_file():
            print(f'  FALTA {disp} -> salteo'); continue
        emb, l2 = embed(loader(ckpt, base), ents)
        rows, eps_star = eps_sweep(emb, y_true=l2, eps_grid=EPS_GRID, min_cluster_size=mcs)
        y0 = HDBSCAN(min_cluster_size=mcs, metric='cosine').fit_predict(emb)
        ys = HDBSCAN(min_cluster_size=mcs, metric='cosine', cluster_selection_epsilon=float(eps_star)).fit_predict(emb)
        rk = rank_metrics(emb[prb], l2[prb], emb[gal], l2[gal])
        km = full_metrics(l2, kmeans_reference(emb, k=cfg['n_true'], seed=SEED))['ARI']
        RESULTS[cfg['name']][disp] = {'eps_star': float(eps_star), 'ari_eps0': full_metrics(l2, y0)['ARI'],
                                      'epsstar': full_metrics(l2, ys), 'rows': rows,
                                      'rank1': round(rk['rank1'], 4), 'kmeans_ari': round(km, 4)}
        p = RESULTS[cfg['name']][disp]
        print(f"  {disp:32} eps*={p['eps_star']:.3f} ARI@eps*={p['epsstar']['ARI']:.3f} noise={p['epsstar']['noise_frac']:.2f}")

(EVALDIR / 'metrics_all.json').write_text(json.dumps(RESULTS, indent=2, default=float))
print('\nmetricas ->', EVALDIR / 'metrics_all.json')


===== A_crudo_mcs4 (crudo (con duplicados), mcs=4 - OPTIMISTA / sin anti-fuga) =====


[16:34:25] INFO encoder='imagenet_resnet50' embedded 4923 imgs → dim 2048
INFO:reid.encoders:encoder='imagenet_resnet50' embedded 4923 imgs → dim 2048


  ImageNet (frozen)                eps*=0.030 ARI@eps*=0.645 noise=0.05


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
[16:37:27] INFO encoder='dinov2-base' embedded 4923 imgs → dim 768
INFO:reid.encoders:encoder='dinov2-base' embedded 4923 imgs → dim 768


  DINOv2-base (frozen)             eps*=0.000 ARI@eps*=0.264 noise=0.13


...

  SupCon (DINOv2-base, strong)     eps*=0.035 ARI@eps*=0.785 noise=0.02


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
[17:01:09] INFO encoder='cmpd300_supcon_dinov2_heavyaug.pt' embedded 1554 imgs → dim 768
INFO:reid.encoders:encoder='cmpd300_supcon_dinov2_heavyaug.pt' embedded 1554 imgs → dim 768


  SupCon (DINOv2-base, heavy)      eps*=0.045 ARI@eps*=0.788 noise=0.02


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
[17:02:14] INFO encoder='cmpd300_supcon_dinov2L_strong.pt' embedded 1554 imgs → dim 1024
INFO:reid.encoders:encoder='cmpd300_supcon_dinov2L_strong.pt' embedded 1554 imgs → dim 1024


  SupCon (DINOv2-large, strong)    eps*=0.050 ARI@eps*=0.835 noise=0.01

metricas -> outputs/results/eval_final/metrics_all.json


## 10. Tablas → Markdown a Drive

Por cada protocolo: **T1** (por loss) y **T2** (por modelo + aug + baselines), batería completa
en eps\*. Más una comparación **A vs B** del ganador.

In [11]:
FULL = [('ARI','{:.3f}'), ('AMI','{:.3f}'), ('NMI','{:.3f}'), ('homogeneity','{:.3f}'),
        ('completeness','{:.3f}'), ('v_measure','{:.3f}'), ('bcubed_P','{:.3f}'),
        ('bcubed_R','{:.3f}'), ('bcubed_F1','{:.3f}'), ('n_clusters','{:d}'),
        ('cluster_ratio','{:.2f}'), ('noise_frac','{:.2f}')]
HEAD = ['ARI(eps0)', 'eps*'] + [c for c, _ in FULL] + ['Rank-1', 'kmeans']

def md_table(cfg_name, names):
    R = RESULTS[cfg_name]
    lines = ['| modelo | ' + ' | '.join(HEAD) + ' |', '|' + '---|' * (len(HEAD) + 1)]
    for nm in names:
        if nm not in R: continue
        r = R[nm]; ms = r['epsstar']
        cells = [f"{r['ari_eps0']:.3f}", f"{r['eps_star']:.3f}"]
        cells += [fmt.format(ms[c]) for c, fmt in FULL]
        cells += [f"{r['rank1']:.3f}", f"{r['kmeans_ari']:.3f}"]
        lines.append('| ' + nm + ' | ' + ' | '.join(cells) + ' |')
    return '\n'.join(lines)

T1n = [n for n in META if META[n]['grupo'] in ('loss', 'both')]
T2n = [n for n in META if META[n]['grupo'] in ('baseline', 'model', 'both')]
docs = []
for cfg in CONFIGS:
    cn, desc = cfg['name'], cfg['desc']
    t1 = f'## T1 [{cn}] por loss — {desc}\n\n' + md_table(cn, T1n)
    t2 = f'## T2 [{cn}] por modelo — {desc}\n\n' + md_table(cn, T2n)
    (EVALDIR / f'T1_{cn}.md').write_text(t1); (EVALDIR / f'T2_{cn}.md').write_text(t2)
    docs += [t1, t2]

# comparacion A vs B del ganador
win = 'SupCon (DINOv2-large, strong)'
cmp_lines = ['## Ganador: A (crudo,mcs4) vs B (dedup,mcs2)', '',
             '| protocolo | ARI@eps0 | ARI@eps* | Rank-1 | noise |', '|---|---|---|---|---|']
for cfg in CONFIGS:
    r = RESULTS[cfg['name']].get(win)
    if r: cmp_lines.append(f"| {cfg['name']} | {r['ari_eps0']:.3f} | {r['epsstar']['ARI']:.3f} | "
                           f"{r['rank1']:.3f} | {r['epsstar']['noise_frac']:.2f} |")
cmp = '\n'.join(cmp_lines); (EVALDIR / 'AvsB_ganador.md').write_text(cmp)
print('\n\n'.join(docs)); print('\n\n', cmp)
print('\ntablas ->', EVALDIR)

## T1 [A_crudo_mcs4] por loss — crudo (con duplicados), mcs=4 - OPTIMISTA / sin anti-fuga

| modelo | ARI(eps0) | eps* | ARI | AMI | NMI | homogeneity | completeness | v_measure | bcubed_P | bcubed_R | bcubed_F1 | n_clusters | cluster_ratio | noise_frac | Rank-1 | kmeans |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| CE | 0.684 | 0.015 | 0.710 | 0.911 | 0.953 | 0.962 | 0.944 | 0.953 | 0.968 | 0.799 | 0.875 | 330 | 1.23 | 0.04 | 0.889 | 0.808 |
| ArcFace | 0.428 | 0.000 | 0.428 | 0.858 | 0.922 | 0.916 | 0.929 | 0.922 | 0.963 | 0.747 | 0.841 | 321 | 1.20 | 0.09 | 0.825 | 0.762 |
| Triplet | 0.414 | 0.005 | 0.421 | 0.854 | 0.921 | 0.918 | 0.924 | 0.921 | 0.961 | 0.728 | 0.829 | 329 | 1.23 | 0.09 | 0.816 | 0.752 |
| SupCon (ResNet, strong) | 0.726 | 0.015 | 0.750 | 0.919 | 0.957 | 0.967 | 0.948 | 0.957 | 0.968 | 0.815 | 0.885 | 325 | 1.21 | 0.03 | 0.890 | 0.831 |

## T2 [A_crudo_mcs4] por modelo — crudo (con duplicados), mcs=4 - OPTIMISTA / sin anti-fuga

| model

## 11. Figura — curva de loss de DINOv2-large (el encoder que usamos)

In [12]:
import re, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
logf = Path('outputs/logs/train_supcon_dinov2L_strong.log')
ep, loss = [], []
if logf.exists():
    for line in logf.read_text().splitlines():
        m = re.search(r'ep (\d+)/\d+ \| loss ([\d.]+)', line)
        if m: ep.append(int(m.group(1))); loss.append(float(m.group(2)))
if ep:
    plt.figure(figsize=(8, 4.5))
    plt.plot(ep, loss, color='#2b6cb0', linewidth=2, marker='o', markersize=3)
    plt.xlabel('Época'); plt.ylabel('SupCon loss (train)')
    plt.title('DINOv2-large + SupCon - curva de entrenamiento', fontweight='bold')
    plt.grid(alpha=0.3); plt.tight_layout()
    out = EVALDIR / 'fig_loss_dinov2L.png'; plt.savefig(out, dpi=150, bbox_inches='tight'); plt.close()
    print(f'curva: {len(ep)} epochs, loss {loss[0]:.3f} -> {loss[-1]:.3f} ->', out)
else:
    print('no encontre outputs/logs/train_supcon_dinov2L_strong.log')

curva: 60 epochs, loss 2.369 -> 1.115 -> outputs/results/eval_final/fig_loss_dinov2L.png


## 12. Figuras — escalera por modelo (protocolo B) + barrido de eps del ganador (B)

In [13]:
from reid_eval import plot_encoder_staircase, plot_eps_sweep
B = 'B_dedup_mcs2'
res_star = {n: RESULTS[B][n]['epsstar'] for n in T2n if n in RESULTS[B]}
plot_encoder_staircase(res_star, metric='ARI', out=str(EVALDIR / 'fig_staircase_B.png'))
win = RESULTS[B].get('SupCon (DINOv2-large, strong)')
if win:
    plot_eps_sweep(win['rows'], eps_star=win['eps_star'], out=str(EVALDIR / 'fig_eps_sweep_B.png'))
print('figuras ->', EVALDIR)

figuras -> outputs/results/eval_final


## 13. Verificación — qué quedó en Drive

In [14]:
for p in sorted(EVALDIR.glob('*')):
    print(f'  {p.name:28} {p.stat().st_size/1024:.0f} KB')
print('\nTODO en', EVALDIR)

  AvsB_ganador.md              0 KB
  T1_A_crudo_mcs4.md           1 KB
  T1_B_dedup_mcs2.md           1 KB
  T2_A_crudo_mcs4.md           1 KB
  T2_B_dedup_mcs2.md           1 KB
  fig_eps_sweep_B.png          98 KB
  fig_loss_dinov2L.png         42 KB
  fig_staircase_B.png          75 KB
  index_dedup.csv              63 KB
  index_raw.csv                200 KB
  metrics_all.json             57 KB
  zenodo_1554_index.csv        63 KB

TODO en outputs/results/eval_final
